#### Why Preprocessing?

```txt
Raw Point Cloud
        │
        ▼
    Noise
    Duplicate
    Outlier
    Huge Data
        │
        ▼
Preprocessing
        │
        ▼
Clean Point Cloud

```



* Goal of SOR - Only remove noise
* Goal of Voxel Sampling - Reduce the number of points by replacing all points inside one representative point

In [95]:
import laspy
import open3d as o3d
import numpy as np

In [96]:
las=laspy.read("sample_data/sample_0000.las")


In [97]:
points=np.column_stack((las.x,las.y,las.z))


In [98]:
pcd=o3d.geometry.PointCloud()

In [99]:
pcd.points=o3d.utility.Vector3dVector(points)

In [100]:
# pcd.paint_uniform_color([0,1,0])

In [101]:
frame=o3d.geometry.TriangleMesh.create_coordinate_frame(size=4.0,origin=[0,0,0])

In [102]:
o3d.visualization.draw_geometries([pcd,frame])

#### Point Cloud Statistics

In [103]:
len(pcd.points)

251164

In [104]:
np.asarray(pcd.points)

array([[-5.31602244e+00, -1.40158173e+00,  2.69150759e-02],
       [-5.30102244e+00, -1.45658173e+00,  1.99150759e-02],
       [-5.29802244e+00, -1.44958173e+00,  1.91507586e-03],
       ...,
       [ 4.60297756e+00,  5.34418274e-01,  2.19150759e-02],
       [ 4.60497756e+00,  5.10418274e-01,  1.89150759e-02],
       [ 4.60597756e+00,  4.90418274e-01, -8.49241370e-05]],
      shape=(251164, 3))

In [105]:
np.asarray(pcd.points).shape

(251164, 3)

In [106]:
pcd.get_max_bound()

array([4.60597756, 4.54841827, 1.43991508])

In [107]:
pcd.get_min_bound()

array([-5.31602244, -5.44558173, -0.19208492])

In [108]:
pcd.get_center()

array([-0.27427717, -0.77000598,  0.2165113 ])

#### Voxel Downsampling

In [109]:
pcd_down=pcd.voxel_down_sample(voxel_size=0.05)

In [110]:
pcd,pcd_down

(PointCloud with 251164 points., PointCloud with 44030 points.)

In [111]:
o3d.visualization.draw_geometries([pcd_down])

#### Uniform Downsampling

In [112]:
pcd_uniform=pcd.uniform_down_sample(every_k_points=5)

In [113]:
pcd,pcd_uniform

(PointCloud with 251164 points., PointCloud with 50233 points.)

In [114]:
o3d.visualization.draw_geometries([pcd_uniform])

#### Compare Different Voxel Size

In [115]:
for voxel in [0.01,0.03,0.05,0.10]:
    down = pcd.voxel_down_sample(voxel)

    print(
        voxel,
        len(down.points)
    )

0.01 241162
0.03 116733
0.05 44030
0.1 8841


### Statistical Outlier Removal

In [116]:
pcd_clean, ind = pcd.remove_statistical_outlier(
    nb_neighbors=20,
    std_ratio=2.0
)

In [117]:
pcd_clean

PointCloud with 251035 points.

In [118]:
o3d.visualization.draw_geometries([pcd_clean])

#### Outliers visualize

In [119]:
outlier_cloud = pcd.select_by_index(
    ind,
    invert=True
)

inlier_cloud = pcd.select_by_index(ind)

In [120]:
outlier_cloud.paint_uniform_color([1,0,0])

inlier_cloud.paint_uniform_color([0.8,0.8,0.8])

PointCloud with 251035 points.

In [121]:
o3d.visualization.draw_geometries(
    [
        inlier_cloud,
        outlier_cloud,
        frame
    ]
)

#### Radius Outlier Removal

```txt
For every point

    ↓

Count neighbors inside radius

    ↓

Enough neighbors?

    ↓

if count==required Yes → Keep

if count!=required No → Remove
```

In [128]:
pcd_radius, ind = pcd.remove_radius_outlier(
    nb_points=16,
    radius=0.1
)

In [129]:
print("Before :",len(pcd.points))
print("After :",len(pcd_radius.points))

Before : 251164
After : 251048


In [130]:
o3d.visualization.draw_geometries(
    [
        pcd_radius,
        frame
    ]
)

#### Compare SOR vs Radius

In [125]:
print("Original :",len(pcd.points))

print("SOR :",len(pcd_clean.points))

print("Radius :",len(pcd_radius.points))

Original : 251164
SOR : 251035
Radius : 251048


#### Crop

In [126]:
bbox = o3d.geometry.AxisAlignedBoundingBox(
    min_bound=(-20,-20,-20),
    max_bound=(20,20,20)
)

crop = pcd.crop(bbox)

In [127]:
o3d.visualization.draw_geometries(
    [
        crop,
        frame
    ]
)

```txt
Raw Point Cloud
        │
        ▼
Voxel Downsample
        │
        ▼
Uniform Downsample
        │
        ▼
       SOR
        │
        ▼
Radius Outlier
        │
        ▼
       Crop
        │
        ▼
Clean Point Cloud
```